## Word Matching with a bipartite weighted graph

In this section, we'll show the matching of our source sentence to the dialect-ignorant transcript (DIT). The alignment is modelled as finding the min-cost max-flow in a bipartite network $G = (W' \cup V', E)$ with source $s$ and sink $t$.

### Dimension
Let $W = \{w_1, \dots, w_m\}$ be the set of source (reference) words and $V = \{v_1, \dots, v_n\}$ be the set of transcribed target words (DIT/DAT). We define the augmented sets $W'$ and $V'$ with cardinality $N = m + n$:
- **Source**: $W' = W \cup \{\epsilon_{w,1}, \dots, \epsilon_{w,n}\}$
- **Target**: $V' = V \cup \{\epsilon_{v,1}, \dots, \epsilon_{v,m}\}$
- **Dimension:** $|W'| = |V'| = N$ (Hall & Frobenius theorems satisfied)

### Cost functions (weights)
The cost $c_{ij}$ for matching an element $i \in W'$ with an element $j \in V'$ is defined as:

$$c_{ij} = \begin{cases} \text{cost}(w_i, v_j) & \text{if } w_i \in W, v_j \in V \\ \lambda & \text{if } (w_i \in W, v_j \notin V) \lor (w_i \notin W, v_j \in V) \\ 0 & \text{if } w_i \notin W, v_j \notin V \end{cases}$$
Where $\text{cost}(w_i, v_j)$ is either just the normalised Levenshtein distance **or** a combined lexical and positional score, 
and $\lambda$ represents the **penalty** for deletions or insertions.
*=> question: where to model position if not in cost*

### Optimisation
Minimise the total cost $Z$ of the flow $f_{ij}$ across all edges:
$$\text{Minimize} \quad Z = \sum_{(i,j) \in E} c_{ij} \cdot f_{ij}$$

#### Constraints
- capacity (one unit of flow)
	- $$f_{ij} \in \{0, 1\} \quad \forall (i, j) \in E$$
- every node $k$ (except source $s$ and sink $t$), incoming flow (i) equals outgoing flow (j):
	- $$\sum_{i:(i,k) \in E} f_{ik} = \sum_{j:(k,j) \in E} f_{kj} \quad \forall k \in V_{graph} \setminus \{s, t\}$$
- total flow must equal the augmented dimension $N$:
	- $$\sum_{j:(s,j) \in E} f_{sj} = N$$

## Imports

In [43]:
import sys
import importlib
sys.path.insert(0, "domain")
import pandas as pd

import bipartite_matching
import calculations
importlib.reload(bipartite_matching)
importlib.reload(calculations)

from bipartite_matching import build_bipartite_graph, solve_matching
from calculations import clean


In [44]:
df_synthetic = pd.read_csv("../transcripts/synthetic/synthetic-dialect-ignorant-transcript.tsv", sep="\t")

row = df_synthetic.iloc[0]
src = clean(row["sentence"]).split()
dit = clean(row["transcript"]).split()

G = build_bipartite_graph(src, dit, max_word_len=1, max_sent_len=5, alpha=1, epsilon_penalty=0.2)
matching = solve_matching(G)
N = len(matching)
n = len(src)
m = len(dit)

# row 1: source
src_row = list(src)

# row 2: hypothesis 1 (matched DIT word per source position, or ε)
hypothesis_row = []
score_row = []
for i in range(n):
    source_word = src[i]
    tgt_node = matching[f"src_{i}"]
    target_word = G.nodes[tgt_node]["word"]
    score = G.edges[f"src_{i}", tgt_node]["score"]
    hypothesis_row.append(target_word)
    score_row.append(f"{score:.2f}")

# row 3: hypothesis 2 (dit): dit in transcribed order
dit_row = list(dit)

# build df
df_alignment = pd.DataFrame(
    [src_row, hypothesis_row, dit_row, score_row],
    index=["Source", "Hypothesis", "DIT", "Score"]
)
df_alignment.columns = range(1, max(n, m) + 1)
df_alignment


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
Source,sie,müssen,so,nahe,wie,möglich,an,das,wasser,heran,nur,gerade,dass,sie,nich,hineinfallen
Hypothesis,sie,ε,so,nach,wie,möglich,ans,ε,wasser,gehen,nur,gerade,dass,sie,nicht,hinein
DIT,sie,munde,so,nach,wie,möglich,ans,wasser,anne,nur,gerade,dass,sie,nicht,hinein,gehen
Score,1.00,0.80,1.00,0.75,1.00,1.00,0.80,0.80,1.00,0.60,1.00,1.00,1.00,1.00,0.89,0.67
